In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
# If charts do not display (for example in VS Code or Colab), uncomment one line:
# pio.renderers.default = "notebook_connected"
# pio.renderers.default = "colab"

print("Plotly version:", __import__("plotly").__version__)

Plotly version: 5.24.1


In [4]:
df = pd.read_csv("dataset for project.csv")
print("Shape:", df.shape)
df.head()

Shape: (3614, 16)


,Student_Name,College_Name,Stream,Year_of_Study,AI_Tools_Used,Daily_Usage_Hours,Use_Cases,Trust_in_AI_Tools,Impact_on_Grades,Do_Professors_Allow_Use,Preferred_AI_Tool,Awareness_Level,Willing_to_Pay_for_Access,State,Device_Used,Internet_Access
0,Aarav,Indian Institute of Information Technology,Engineering,4,Gemini,0.9,"Assignments, Coding Help",2,2,No,Copilot,9,Yes,Uttar pradesh,Mobile,Poor
1,Vivaan,"Government Ram Bhajan Rai NES College, Jashpur",Commerce,2,ChatGPT,3.4,Learning new topics,3,-3,Yes,Other,6,No,Chhattisgarh,Laptop,Poor
2,Aditya,Dolphin PG Institute of BioMedical & Natural,Science,2,Copilot,3.6,"MCQ Practice, Projects",5,0,No,Gemini,1,No,Uttarakhand,Tablet,Poor
3,Vihaan,Shaheed Rajguru College of Applied Sciences for,Arts,2,Copilot,2.9,Content Writing,5,2,Yes,Gemini,5,No,Delhi ncr,Laptop,High
4,Arjun,Roorkee College of Engineering,Science,1,Gemini,0.9,"Doubt Solving, Resume Writing",1,3,Yes,Other,8,Yes,Uttarakhand,Laptop,Medium


In [5]:
# Data types and missing values per column
summary = pd.DataFrame({"dtype": df.dtypes.astype(str),
                        "missing": df.isna().sum(),
                        "missing_%": (df.isna().mean() * 100).round(1)})
summary

,dtype,missing,missing_%
Student_Name,object,0,0.0
College_Name,object,0,0.0
Stream,object,0,0.0
Year_of_Study,int64,0,0.0
AI_Tools_Used,object,0,0.0
Daily_Usage_Hours,float64,0,0.0
Use_Cases,object,0,0.0
Trust_in_AI_Tools,int64,0,0.0
Impact_on_Grades,int64,0,0.0
Do_Professors_Allow_Use,object,0,0.0


In [6]:
# Summary statistics for the numerical survey columns
df[["Year_of_Study", "Daily_Usage_Hours", "Trust_in_AI_Tools",
    "Impact_on_Grades", "Awareness_Level"]].describe().round(2)

,Year_of_Study,Daily_Usage_Hours,Trust_in_AI_Tools,Impact_on_Grades,Awareness_Level
count,3614.00,3614.00,3614.00,3614.00,3614.00
mean,2.47,2.56,3.02,0.00,5.83
std,1.11,1.21,1.44,2.37,2.93
min,1.00,0.50,1.00,-5.00,1.00
25%,2.00,1.50,2.00,-2.00,3.00
50%,2.00,2.60,3.00,0.00,6.00
75%,3.00,3.60,4.00,2.00,8.00
max,4.00,5.00,5.00,5.00,10.00


In [7]:
# Clean text columns and handle missing state values
text_columns = ["Student_Name", "College_Name", "Stream", "AI_Tools_Used",
                "Use_Cases", "Do_Professors_Allow_Use", "Preferred_AI_Tool",
                "Willing_to_Pay_for_Access", "State", "Device_Used", "Internet_Access"]

for col in text_columns:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# Keep the study-year order consistent across charts
YEAR_ORDER = sorted(df["Year_of_Study"].dropna().unique().tolist())

# Fixed category order and colours reused in charts
TOOL_ORDER = df["Preferred_AI_Tool"].value_counts().index.tolist()
TOOL_COLORS = px.colors.qualitative.Plotly
TOOL_COLOR_MAP = {tool: TOOL_COLORS[i % len(TOOL_COLORS)]
                  for i, tool in enumerate(TOOL_ORDER)}

df = df.reset_index(drop=True)
print("Rows and columns after cleaning:", df.shape)
print("Missing values remaining:", int(df.isna().sum().sum()))
print("Study years:", YEAR_ORDER)
print("Preferred AI tools:", df["Preferred_AI_Tool"].value_counts())

Rows and columns after cleaning: (3614, 16)
Missing values remaining: 0
Study years: [1, 2, 3, 4]
Preferred AI tools: Preferred_AI_Tool
ChatGPT    859
Gemini     854
Copilot    836
Other      757
Claude     157
Bard       151
Name: count, dtype: int64


In [8]:
# Quick check: number of students in each year of study
df.groupby("Year_of_Study").size().rename("students").to_frame().T

Year_of_Study,1,2,3,4
students,902,966,881,865


In [9]:
fig1 = px.histogram(df, x="Daily_Usage_Hours",
                    title="Distribution of Daily AI Tool Usage",
                    labels={"Daily_Usage_Hours": "Daily usage (hours)"})
fig1.update_layout(yaxis_title="Number of students", bargap=0.05,
                   xaxis=dict(rangeslider=dict(visible=True, thickness=0.12)))
fig1.show()

In [10]:
fig2 = px.scatter(df, x="Daily_Usage_Hours", y="Awareness_Level",
                  color="Preferred_AI_Tool",
                  color_discrete_map=TOOL_COLOR_MAP,
                  hover_data={"Stream": True, "Year_of_Study": True,
                              "Trust_in_AI_Tools": True, "Impact_on_Grades": True},
                  opacity=0.70,
                  labels={"Daily_Usage_Hours": "Daily usage (hours)",
                          "Awareness_Level": "AI awareness level",
                          "Preferred_AI_Tool": "Preferred AI tool"},
                  title="Daily AI Tool Usage vs. AI Awareness")
fig2.show()

In [11]:
fig3 = px.box(df, x="Preferred_AI_Tool", y="Daily_Usage_Hours",
              color="Preferred_AI_Tool",
              color_discrete_map=TOOL_COLOR_MAP,
              points="all",
              hover_data={"Year_of_Study": True, "Stream": True,
                          "Awareness_Level": True, "Trust_in_AI_Tools": True},
              labels={"Preferred_AI_Tool": "Preferred AI tool",
                      "Daily_Usage_Hours": "Daily usage (hours)"},
              title="Daily AI Tool Usage by Preferred AI Tool (all records shown)")
fig3.update_traces(marker=dict(size=3, opacity=0.45), jitter=0.35)
fig3.update_layout(showlegend=False, xaxis_tickangle=-25)
fig3.show()

In [12]:
# Numbers behind the box plot
df.groupby("Preferred_AI_Tool")[["Daily_Usage_Hours", "Trust_in_AI_Tools",
                                  "Awareness_Level"]].median().reindex(TOOL_ORDER).round(2)

,Daily_Usage_Hours,Trust_in_AI_Tools,Awareness_Level
Preferred_AI_Tool,,,
ChatGPT,2.6,3.0,6.0
Gemini,2.7,3.0,6.0
Copilot,2.6,3.0,6.0
Other,2.3,3.0,6.0
Claude,3.0,3.0,6.0
Bard,2.9,3.0,6.0


In [13]:
fig4 = px.scatter_3d(df, x="Daily_Usage_Hours", y="Trust_in_AI_Tools",
                      z="Awareness_Level", color="Preferred_AI_Tool",
                      color_discrete_map=TOOL_COLOR_MAP,
                      hover_data=["Stream", "Year_of_Study", "Impact_on_Grades"],
                      opacity=0.70,
                      labels={"Daily_Usage_Hours": "Daily usage (hours)",
                              "Trust_in_AI_Tools": "Trust in AI tools (1–5)",
                              "Awareness_Level": "AI awareness level",
                              "Preferred_AI_Tool": "Preferred AI tool"},
                      title="Daily Usage, Trust and Awareness in 3D (rotate and zoom)")
fig4.update_traces(marker=dict(size=3))
fig4.update_layout(height=650)
fig4.show()

In [14]:
fig5 = px.scatter(df, x="Daily_Usage_Hours", y="Awareness_Level",
                  color="Preferred_AI_Tool", size="Trust_in_AI_Tools",
                  animation_frame="Year_of_Study",
                  category_orders={"Year_of_Study": YEAR_ORDER,
                                   "Preferred_AI_Tool": TOOL_ORDER},
                  color_discrete_map=TOOL_COLOR_MAP,
                  hover_data={"Stream": True, "Impact_on_Grades": True,
                              "Year_of_Study": False},
                  size_max=18, opacity=0.70,
                  range_x=[0, max(5, float(df["Daily_Usage_Hours"].max()))],
                  range_y=[0, 10],
                  labels={"Daily_Usage_Hours": "Daily usage (hours)",
                          "Awareness_Level": "AI awareness level",
                          "Preferred_AI_Tool": "Preferred AI tool",
                          "Year_of_Study": "Year of study"},
                  title="Daily Usage vs. Awareness by Year of Study (press play); marker size = trust")
fig5.update_layout(legend_itemsizing="constant")
if fig5.layout.updatemenus:
    fig5.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 900
fig5.show()

In [15]:
# Number of student records in each year of study
df.groupby("Year_of_Study").size().rename("students").to_frame().T

Year_of_Study,1,2,3,4
students,902,966,881,865


In [16]:
# The dropdown keeps daily usage on the x-axis and switches the y-axis metric.
metrics = {
    "Awareness": ("AI awareness level", "linear"),
    "Trust": ("Trust in AI tools (1–5)", "linear"),
    "Grade impact": ("Perceived impact on grades", "linear"),
    "Study year": ("Year of study", "linear"),
}

colors = df["Preferred_AI_Tool"].map(TOOL_COLOR_MAP)
initial_y = df["Awareness_Level"]

fig6 = go.Figure(go.Scatter(
    x=df["Daily_Usage_Hours"], y=initial_y, mode="markers",
    marker=dict(color=colors, size=7, opacity=0.7,
                line=dict(width=0.5, color="white")),
    text=df["Preferred_AI_Tool"],
    customdata=np.stack([df["Stream"].astype(str),
                         df["Year_of_Study"].astype(str)], axis=-1),
    hovertemplate="<b>%{text}</b><br>Daily usage: %{x} hours<br>Value: %{y}<br>Stream: %{customdata[0]}<br>Year of study: %{customdata[1]}<extra></extra>",
))

metric_columns = {
    "Awareness": "Awareness_Level",
    "Trust": "Trust_in_AI_Tools",
    "Grade impact": "Impact_on_Grades",
    "Study year": "Year_of_Study",
}
buttons = []
for label, col in metric_columns.items():
    buttons.append(dict(
        label=label,
        method="update",
        args=[{"y": [df[col]]},
              {"yaxis.title.text": metrics[label][0],
               "yaxis.type": metrics[label][1],
               "title.text": f"Daily AI Tool Usage vs. {label}"}],
    ))

fig6.update_layout(
    title="Daily AI Tool Usage vs. AI awareness level",
    xaxis_title="Daily usage (hours)",
    yaxis_title=metrics["Awareness"][0],
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0,
                      xanchor="right", y=1.15, yanchor="top", showactive=True)],
    annotations=[dict(text="y-axis:", x=0.90, xref="paper", y=1.12,
                      yref="paper", showarrow=False, xanchor="right")],
)
fig6.show()

In [17]:
# Spearman correlations between the main numerical survey measures
df[["Daily_Usage_Hours", "Trust_in_AI_Tools", "Impact_on_Grades",
    "Awareness_Level", "Year_of_Study"]].corr(method="spearman").round(2)

,Daily_Usage_Hours,Trust_in_AI_Tools,Impact_on_Grades,Awareness_Level,Year_of_Study
Daily_Usage_Hours,1.00,-0.01,0.06,0.00,0.01
Trust_in_AI_Tools,-0.01,1.00,-0.05,0.04,0.00
Impact_on_Grades,0.06,-0.05,1.00,-0.01,-0.01
Awareness_Level,0.00,0.04,-0.01,1.00,-0.01
Year_of_Study,0.01,0.00,-0.01,-0.01,1.00


In [18]:
figs = [fig1, fig2, fig3, fig4, fig5, fig6]
with open("Lab6_AI_Tools_Interactive_Charts.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Lab 6: AI Tools Interactive Charts</title></head><body>")
    for i, fig in enumerate(figs):
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False,
                            auto_play=False))
    f.write("</body></html>")
print("Saved Lab6_AI_Tools_Interactive_Charts.html")

Saved Lab6_AI_Tools_Interactive_Charts.html
